In [ ]:
import numpy as np
import xarray as xr
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
def angle_to_l(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 180. / x[~near_zero]
    return x


In [ ]:
def freq_to_time(x):
    """Vectorized 1/x, treating x==0 manually"""
    x = np.array(x, float)
    near_zero = np.isclose(x, 0)
    x[near_zero] = np.inf
    x[~near_zero] = 1. / x[~near_zero]
    return x


In [ ]:
# grid
ntim = 365*2*30
ntrunc=72
spd = 2
frequency = np.fft.fftfreq(ntim, 1./spd)
ff = np.fft.fftshift(frequency[:])
ll = np.arange(0, ntrunc+1, 1)

nSampWin_360 = 720
nWindow_360 = 59
frequency_360 = np.fft.fftfreq(nSampWin_360, 1./spd)
ff_360 = np.fft.fftshift(frequency_360[:])


In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
# raw spectra
file_name = "olr-2xdaily-1981-2010-space-time-analysis-raw.nc"
ds_raw = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fflm_raw = ds_raw.Fflm_real.values[:, :, :] + 1j * ds_raw.Fflm_imag.values[:, :, :]
Ftlm_raw = ds_raw.Ftlm_real.values[:, :, :] + 1j * ds_raw.Ftlm_imag.values[:, :, :]

In [ ]:
Fflm_raw = np.fft.fftshift(Fflm_raw, axes=0)

In [ ]:
# angular variance -- raw
Cl_raw = np.zeros(ntrunc+1)
Cl_raw_std = np.zeros(ntrunc+1)

for l in range(ntrunc+1):
    Cl_raw[l] = np.mean(np.mean(np.abs(Ftlm_raw[:, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0)
    Cl_raw_std[l] = np.std(np.std(np.abs(Ftlm_raw[:, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0)


In [ ]:
# temporal variance -- raw
Cfl_raw = np.zeros((ntim, ntrunc+1))
Cfl_raw_std = np.zeros((ntim, ntrunc+1))

for l in range(2, ntrunc+1):
    Cfl_raw[:, l] = np.mean(np.abs(Fflm_raw[:, l, ntrunc-l:ntrunc+l+1])**2, axis=1)
    Cfl_raw_std[:, l] = np.std(np.abs(Fflm_raw[:, l, ntrunc-l:ntrunc+l+1])**2, axis=1)

Cf_raw = np.mean(Cfl_raw, axis=1)
Cf_raw_std = np.std(Cfl_raw_std, axis=1)

# for psd need to divide by dw = frequency resolution = sample rate / number of samples
Cf_raw *= ntim / 2.
Cf_raw_std *= ntim / 2.


In [ ]:
# processed data
file_name = "olr-2xdaily-1981-2010-space-time-analysis-window-360-skip-180.nc"
ds_360 = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
Fwflm_360 = ds_360.Fwflm_real.values[:, :, :, :] + 1j * ds_360.Fwflm_imag.values[:, :, :, :]
Fwtlm_360 = ds_360.Fwtlm_real.values[:, :, :, :] + 1j * ds_360.Fwtlm_imag.values[:, :, :, :]

In [ ]:
Fwflm_360 = np.fft.fftshift(Fwflm_360, axes=1)

In [ ]:
# angular variance -- 360
Cl_360 = np.zeros(ntrunc+1)

for l in range(ntrunc+1):
    Cl_360[l] = np.mean(np.mean(np.mean(np.abs(Fwtlm_360[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=0), axis=0)

# compensating for hanning window
Cl_360 *= 8. / 3.


In [ ]:
# temporal variance -- 360
Cfl_360 = np.zeros((nSampWin_360, ntrunc+1))

for l in range(2, ntrunc+1):
    Cfl_360[:, l] = np.mean(np.mean(np.abs(Fwflm_360[:, :, l, ntrunc-l:ntrunc+l+1])**2, axis=0), axis=1)

Cf_360 = np.mean(Cfl_360, axis=1)

# compensating for hanning window
Cf_360 *= 8. / 3.

# for psd need to divide by dw = frequency resolution = sample rate / number of samples
Cf_360 *= nSampWin_360 / 2.


In [ ]:
# because the variance changes by 2 orders of magnitudes it's better to fit the scaled data

In [ ]:
# cmb scaling
for l in range(ntrunc+1):
    Cl_raw[l] = l*(l+1) / (2*np.pi) * Cl_raw[l]
    Cl_360[l] = l*(l+1) / (2*np.pi) * Cl_360[l]

In [ ]:
# fit angular variance
# a**2 / (1 + b**2 * l(l+1))**2 * l*(l+1) / 2pi

popt_l, pcov_l = curve_fit(lambda l, a, b: a**2 * l * (l+1) / (2*np.pi) / ( 1 + b**2 * l * (l+1) )**2, ll[2:], Cl_360[2:], p0=[5.5, 0.05])
perr_l = np.sqrt(np.diag(pcov_l))

Cl_ebcm = popt_l[0]**2 * ll * (ll+1) / (2*np.pi) / ( 1 + popt_l[1]**2 * ll * (ll+1) )**2

print(popt_l[0], perr_l[0])
print(popt_l[1], perr_l[1])

In [ ]:
# fit temporal
# a**2 * b / (1 + b**2 * f**2)**2

# need to take the mean over l of the formula in the text 
def temporal_variance(f, epsilon_0, lambda_0, tau_0):
    temporal_variance_l = np.zeros((f.shape[0], ntrunc+1))
    for l in range(2, ntrunc+1):
        tau_l = tau_0 / (1 + lambda_0**2 * l * (l+1))
        epsilon_l = epsilon_0 * tau_l / tau_0
        temporal_variance_l[:, l] =  2 * epsilon_l**2 * tau_l / (1 + tau_l**2 * f**2 * 4 * np.pi**2)
    return np.mean(temporal_variance_l, axis=1)

# fit only on subannual timescales
ind = np.where((ff_360 > 1./360) & (ff_360 < 1))[0]

popt_w, pcov_w = curve_fit(lambda f, c: temporal_variance(f, epsilon_0=popt_l[0], lambda_0=popt_l[1], tau_0=c), ff_360[ind], Cf_360[ind], p0=[2.5])
perr_w = np.sqrt(np.diag(pcov_w))

# plot on all scales
ind_fit_w = np.where((ff >= 0) & (ff <= 1))[0]

Cf_ebcm = temporal_variance(ff[ind_fit_w], popt_l[0], popt_l[1], popt_w[0])

print(popt_w[0], perr_w[0])
# print(popt_w[1], perr_w[1])
# print(popt_w[2], perr_w[2])

In [ ]:
fig_size = (14.20/2.54, 08.50/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 01.25, 12.00, 06.00])))

##### ax0 #####

ax[0].plot(ll, Cl_raw, linestyle='', marker='.', markersize=4, color='C1', label='Unfiltered', alpha=1)
ax[0].plot(ll, Cl_360, linestyle='', marker='.', markersize=4, fillstyle='full', color='C0', label='Subannual', alpha=1)

# standard error in the variance over windows and over m 
ax[0].fill_between(ll, (Cl_raw + Cl_raw * (2/(2*ll+1)/1)**0.5), (Cl_raw - Cl_raw * (2/(2*ll+1)/2)**0.5), color='C1', alpha=0.2) #, label='Standard\nerror')
ax[0].fill_between(ll, (Cl_360 + Cl_360 * (2/(2*ll+1)/58)**0.5), (Cl_360 - Cl_360 * (2/(2*ll+1)/58)**0.5), color='C0', alpha=0.2) #, label='Standard\nerror')

# ebcm
ax[0].plot(ll, Cl_ebcm, linestyle='-', linewidth=1, marker='', color='black', label=r'$\mathcal{E}\,$BCM')

secax = ax[0].secondary_xaxis('top', functions=(angle_to_l, angle_to_l))
secax.set_xlabel('Angular scale', fontsize=10)
# secax.set_xticks(np.array([90, 20, 10, 7, 6, 5, 4, 3]))
secax.set_xticks(np.array([18, 9, 6, 4.5, 3.6, 3]))
secax.xaxis.set_major_formatter(StrMethodFormatter(u"{x:.1f}°"))
secax.tick_params(axis='both', which='both', labelsize=10)
secax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')

ax[0].legend(ncol=1, fontsize=10, frameon=True, edgecolor='grey')
# ax[0].set_title('Angular power spectrum', fontweight='bold')
ax[0].set_xlabel(r'Multipole moment ($\ell$)', fontsize=10)
ax[0].set_ylabel(r'$\ell (\ell+1) \, C_{\ell} \, / \, 2\pi$  $\mathrm{ [W \, m^{-2}]^{2}}$', fontsize=10)
ax[0].tick_params(axis='both', which='both', labelsize=10)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
# ax[0].set_xscale('log')
# ax[0].set_yscale('log')
# ax[0].set_xlim(1, 80)
ax[0].set_ylim(None, 600)

# ax[0].grid()

string = (r'$ \epsilon_{0} = $' + r'$ {epsilon:.1f} \pm {err0:.1f} $'.format(epsilon=abs(popt_l[0]), err0=abs(perr_l[0])) + r' $ \mathrm{ [W m^{-2}]} $' +
          '\n' +
         r'$ \lambda_{0} = $' + r'$ {lambda0:3.0f} \pm {err0:2.0f} $'.format(lambda0=6371*abs(popt_l[1]), err0=6371*abs(perr_l[1])) + r' $ \mathrm{ [km]} $'
         )

t1 = ax[0].text(00.66, 00.59, string, ha='left', va='top', transform=ax[0].transAxes, fontsize=10, backgroundcolor='white')
t1.set_bbox(dict(facecolor='white', alpha=0.8, edgecolor='grey'))


In [ ]:
file_name = "fig-02"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')

In [ ]:
fig_size = (14.20/2.54, 08.50/2.54)
fig = plt.figure(figsize=fig_size)

ax = []

ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.75, 01.25, 12.00, 06.00])))


##### ax0 #####

ax[0].plot(ff, Cf_raw, linestyle='', marker='.', markersize=4, color='C1', alpha=1, label='Unfiltered')
ax[0].plot(ff_360[ff_360 >= 1/360], Cf_360[ff_360 >= 1/360], linestyle='', marker='.', markersize=4, fillstyle='full', color='C0', alpha=1, label='Subannual')

ax[0].fill_between(ff, Cf_raw*(10**(0.434*Cf_raw*(2/(2*0+1)/2)**0.5/Cf_raw)), Cf_raw/(10**(0.434*Cf_raw*(2/(2*0+1)/2)**0.5/Cf_raw)), color='C1', alpha=0.2)
ax[0].fill_between(ff_360[ff_360 >= 1/360],
                   Cf_360[ff_360 >= 1/360]*(10**(0.434*Cf_360[ff_360 >= 1/360]*(2/(2*0+1)/58)**0.5/Cf_360[ff_360 >= 1/360])),
                   Cf_360[ff_360 >= 1/360]/(10**(0.434*Cf_360[ff_360 >= 1/360]*(2/(2*0+1)/58)**0.5/Cf_360[ff_360 >= 1/360])), color='C0', alpha=0.2)


ax[0].plot(ff[ind_fit_w], Cf_ebcm, linestyle='-', linewidth=1, marker='', color='black', label=r'$\mathcal{E}\,$BCM')


secax = ax[0].secondary_xaxis('top', functions=(freq_to_time, freq_to_time))
secax.set_xlabel('Period [day]', fontsize=10)
# secax.set_xticks(np.array([90, 0, 60, 30, 10, 5, 2]))
secax.tick_params(axis='both', which='both', labelsize=10)
secax.tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
secax.tick_params(axis='both', which='minor', direction='out', size=2, color='0.8')

ax[0].legend(ncol=1, loc='lower left', fontsize=10, frameon=True, edgecolor='grey')
# ax[0].set_title('Temporal power spectrum', fontweight='bold')
ax[0].set_xlabel(r'Frequency ($\omega$) [cpd]', fontsize=10)
ax[0].set_ylabel(r'$\hat{C}$ $\mathrm{ [W \, m^{-2}]^{2} \, day}$', fontsize=10)
ax[0].tick_params(axis='both', which='both', labelsize=10)
ax[0].tick_params(axis='both', which='major', direction='out', size=3.5, color='0.8')
ax[0].tick_params(axis='both', which='minor', direction='out', size=2, color='0.8')
ax[0].yaxis.set_label_coords(-0.1, 0.5)
ax[0].set_xscale('log')
ax[0].set_yscale('log')
# ax[0].set_xlim(1./360, None)
# ax[0].set_ylim(1e-1, 1e2)
ax[0].set_ylim(1e-1, 1e4)
# ax[0].set_xlim(3./370, 3./350)

ax[0].axvline(x=1./365, linestyle='--', marker='', color='grey', lw=0.5)
ax[0].axvline(x=2./365, linestyle='--', marker='', color='grey', lw=0.5)
ax[0].axvline(x=3./365, linestyle='--', marker='', color='grey', lw=0.5)
ax[0].axvline(x=4./365, linestyle='--', marker='', color='grey', lw=0.5)

ind3 = np.argmin(np.abs(ff - 3./365))

ax[0].plot(ff[ff == 1./365], Cf_raw[ff == 1./365], linestyle='', marker='.', markersize=6, color='C1', alpha=1)
ax[0].plot(ff[ff == 2./365], Cf_raw[ff == 2./365], linestyle='', marker='.', markersize=6, color='C1', alpha=1)
ax[0].plot(ff[ind3], Cf_raw[ind3], linestyle='', marker='.', markersize=6, color='C1', alpha=1)
ax[0].plot(ff[ff == 4./365], Cf_raw[ff == 4./365], linestyle='', marker='.', markersize=6, color='C1', alpha=1)

string = (r'$ \tau_{0} = $' + r'$ {tau:1.1f} \pm {err0:0.1f} $'.format(tau=abs(popt_w[0]), err0=perr_w[0]) + r' $ \mathrm{ [day]} $')

t1 = ax[0].text(00.63, 00.95, string, ha='left', va='top', transform=ax[0].transAxes, fontsize=10, backgroundcolor='white')
t1.set_bbox(dict(facecolor='white', alpha=0.8, edgecolor='grey'))

# ax[0].grid()


In [ ]:
file_name = "fig-03"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')